# 8.3 Datos para exportar

In [75]:
roc_rows, pr_rows = [], []

for nombre, r in resultados_bin.items():
    fpr, tpr, _ = roc_curve(y_te, r['proba'])
    for f, t in zip(fpr, tpr):
        roc_rows.append({'modelo': f"{nombre} (AUC={r['auc']:.3f})", 'fpr': f, 'tpr': t})

    prec, rec, _ = precision_recall_curve(y_te, r['proba'])
    ap = average_precision_score(y_te, r['proba'])
    for p, rr in zip(prec, rec):
        pr_rows.append({'modelo': f"{nombre} (AP={ap:.3f})", 'recall': rr, 'precision': p})

# Línea de referencia diagonal para ROC
roc_rows += [{'modelo': 'Aleatorio', 'fpr': 0, 'tpr': 0},
             {'modelo': 'Aleatorio', 'fpr': 1, 'tpr': 1}]

# Línea base para Precisión-Recall
baseline_pr = y_te.mean()
pr_rows += [{'modelo': f'Baseline ({baseline_pr:.2f})', 'recall': 0, 'precision': baseline_pr},
            {'modelo': f'Baseline ({baseline_pr:.2f})', 'recall': 1, 'precision': baseline_pr}]

pd.DataFrame(roc_rows).to_csv('roc_curve_3modelos.csv', index=False)
pd.DataFrame(pr_rows).to_csv('pr_curve_3modelos.csv', index=False)

# Matriz de confusión — del mejor modelo (mejor_nombre)
df_confusion = pd.DataFrame({
    'cod_mun': test['cod_mun'].values,
    'real': y_te.values,
    'prediccion': mejor['preds']
})
df_confusion.to_csv('confusion_matrix.csv', index=False)

print(f"Mejor modelo exportado en matriz de confusión: {mejor_nombre}")

Mejor modelo exportado en matriz de confusión: Random Forest


In [76]:
df_shap_export = importancia_shap.head(15).copy()
df_shap_export['feature_label'] = df_shap_export['feature'].map(lambda f: ETIQ.get(f, f))

df_shap_export[['feature_label', 'shap_abs_medio']].to_csv('shap_importance.csv', index=False
)

In [77]:
import pandas as pd
import numpy as np

def preparar_waterfall(shap_vals, base_value, features, top_n=10):
    df = pd.DataFrame({'feature': features, 'shap_value': shap_vals})
    df['abs_shap'] = df['shap_value'].abs()
    df = df.sort_values('abs_shap', ascending=False).reset_index(drop=True)

    if len(df) > top_n:
        resto = df.iloc[top_n:]['shap_value'].sum()
        df = df.iloc[:top_n].copy()
        df.loc[len(df)] = ['Otras variables', resto, abs(resto)]

    filas, acumulado, orden = [], base_value, 0
    filas.append({'orden': orden, 'feature': 'Valor base', 'start': 0,
                   'delta': base_value, 'tipo': 'base'})
    orden += 1

    for _, row in df.iterrows():
        filas.append({'orden': orden, 'feature': row['feature'], 'start': acumulado,
                       'delta': row['shap_value'],
                       'tipo': 'positivo' if row['shap_value'] >= 0 else 'negativo'})
        acumulado += row['shap_value']
        orden += 1

    filas.append({'orden': orden, 'feature': 'Predicción final', 'start': 0,
                   'delta': acumulado, 'tipo': 'total'})
    return pd.DataFrame(filas)

# ── Waterfall global (promedio de todo el test set) ──
shap_medio_signed  = shap_values_rf.values.mean(axis=0)
base_value_global  = shap_values_rf.base_values.mean()

df_waterfall_global = preparar_waterfall(shap_medio_signed, base_value_global, X_te.columns.tolist())
df_waterfall_global['feature_label'] = df_waterfall_global['feature'].map(lambda f: ETIQ.get(f, f))
df_waterfall_global.to_csv('shap_waterfall_global.csv', index=False)

# ── "Force plot" — casos locales (reutiliza tus dos ejemplos ya calculados) ──
casos = [('Mayor riesgo predicho', idx_prioritario, 'alto'),
         ('Menor riesgo predicho', idx_no_prioritario, 'bajo')]

dfs_local = []
for nombre_caso, idx, _ in casos:
    shap_fila = shap_values_rf.values[idx]
    bv = shap_values_rf.base_values[idx] if hasattr(shap_values_rf.base_values, '__len__') else shap_values_rf.base_values
    df_local = preparar_waterfall(shap_fila, bv, X_te.columns.tolist())
    df_local['feature_label'] = df_local['feature'].map(lambda f: ETIQ.get(f, f))
    df_local['caso'] = nombre_caso
    df_local['cod_mun'] = test.iloc[idx]['cod_mun']
    dfs_local.append(df_local)

pd.concat(dfs_local, ignore_index=True).to_csv('shap_force_casos.csv', index=False)

In [78]:
chequeo = pd.read_csv('shap_force_casos.csv')
print(chequeo.columns.tolist())
print(chequeo.shape)
chequeo.head()

['orden', 'feature', 'start', 'delta', 'tipo', 'feature_label', 'caso', 'cod_mun']
(26, 8)


,orden,feature,start,delta,tipo,feature_label,caso,cod_mun
0,0,Valor base,0.000000,0.500535,base,Valor base,Mayor riesgo predicho,5002
1,1,tasa_lag1,0.500535,0.066181,positivo,tasa_lag1,Mayor riesgo predicho,5002
2,2,media_movil_2a,0.566716,0.054591,positivo,media_movil_2a,Mayor riesgo predicho,5002
3,3,longitud,0.621307,-0.036408,negativo,longitud,Mayor riesgo predicho,5002
4,4,tasa_lag2,0.584899,-0.030922,negativo,tasa_lag2,Mayor riesgo predicho,5002


In [79]:
print(shap_values_rf.base_values.shape)   # debería ser (n,) — un solo número por fila
print(shap_values_rf.base_values[:5])     # deberían verse valores cercanos a 0.30-0.35
print(base_value_global)                  # el promedio que exportaste — ¿da cerca de 0.34?

(1566,)
[0.50053519 0.50053519 0.50053519 0.50053519 0.50053519]
0.5005351934404779


In [80]:
shap.plots.force(
    shap_values_rf.base_values[idx_prioritario],
    shap_values_rf.values[idx_prioritario],
    X_te.iloc[idx_prioritario],
    matplotlib=True,
    show=False
)
plt.savefig('force_plot_alto.png', dpi=150, bbox_inches='tight')